# Beverage sales business questions
Read the physical Gold summaries produced by 01_process_gold.

In [ ]:
import importlib
import sys
from pathlib import Path

from pyspark.sql import Window
from pyspark.sql import functions as F

project_root = Path(dbutils.widgets.getAll().get(
    "project_root", "/Workspace/Users/allanjs3@yahoo.com.br/beverage_sales"
))
if not project_root.is_absolute() or not (project_root / "configs/gold").is_dir():
    raise ValueError("Project Root must contain configs/gold.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
importlib.reload(importlib.import_module("utils.config"))
from utils.config import load_configs


## Parameters
Identify the project configuration.

In [ ]:
dbutils.widgets.text("project_root", str(project_root), "Project Root")
tables = load_configs(project_root / "configs/gold")


## 4.1 Top three trade groups per region
Rank by total sales descending, then trade group name to resolve ties.

In [ ]:
trade_groups = spark.table(tables["agg_sales_region_trade_group"]["target"])
group_ranking = Window.partitionBy("region_name").orderBy(
    F.desc("sales_amount"), F.asc("trade_group_description")
)
top_groups = (
    trade_groups.withColumn("rank", F.row_number().over(group_ranking))
    .filter("rank <= 3").orderBy("region_name", "rank")
)
display(top_groups)


## 4.2 Monthly sales by brand
Keep each calendar year and month separate.

In [ ]:
monthly_sales = (
    spark.table(tables["agg_sales_brand_month"]["target"])
    .orderBy("year", "month", "brand_name")
)
display(monthly_sales)


## 4.3 Lowest-selling brands per region
Include all brands tied at the minimum. Missing combinations are not treated as zero sales.

In [ ]:
regional_brands = spark.table(tables["agg_sales_region_brand"]["target"])
brand_ranking = Window.partitionBy("region_name").orderBy(F.asc("sales_amount"))
lowest_sales = (
    regional_brands.withColumn("rank", F.dense_rank().over(brand_ranking))
    .filter("rank = 1").orderBy("region_name", "brand_name")
)
display(lowest_sales)
